# Capítulo 06: Redes Recorrentes - Soluções dos Exercícios

**Livro:** Análise de Dados e Previsão de Séries Temporais com ML e Sistemas Híbridos Inteligentes

Este notebook contém as soluções completas dos exercícios propostos no Capítulo 06, cobrindo:
- LSTM com diferentes tamanhos de janela
- Comparação LSTM vs GRU
- Arquiteturas empilhadas (stacked)
---

In [ ]:
# Configuração inicial e imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy import stats
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import time
import warnings
warnings.filterwarnings('ignore')

# Configuração para reprodutibilidade
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
print('Ambiente configurado com sucesso!')

---

## Exercício 6.1: LSTM com Diferentes Tamanhos de Janela

**Enunciado:** Implemente uma LSTM em PyTorch para prever o Índice Bovespa usando diferentes tamanhos de janela de entrada (10, 20, 30, 50, 100 dias). Para cada configuração, treine o modelo e avalie o RMSE em previsões 5 dias à frente. Existe um tamanho ótimo de janela?

In [ ]:
# =============================================================================
# EXERCÍCIO 6.1: LSTM com Diferentes Tamanhos de Janela
# =============================================================================

# Simulação de dados do IBOVESPA
np.random.seed(42)
dias = pd.date_range('2018-01-01', '2023-12-31', freq='D')
n = len(dias)

returns = np.random.normal(0.0003, 0.015, n)
prices = 100000 * np.exp(np.cumsum(returns))

df_bovespa = pd.DataFrame({
    'close': prices
}, index=dias)

print(f'Dados Bovespa: {len(df_bovespa)} dias')
print(df_bovespa.head())

In [ ]:
# Classe LSTM
class LSTM_Model(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, output_size=5):
        super(LSTM_Model, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2 if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]
        return self.fc(last_hidden)

# Função para criar sequências
def create_sequences(data, n_lags, horizon=5):
    X, y = [], []
    for i in range(n_lags, len(data) - horizon + 1):
        X.append(data[i-n_lags:i])
        y.append(data[i:i+horizon])
    return np.array(X), np.array(y)

# Função de treinamento
def train_lstm(X_train, y_train, X_val, y_val, n_epochs=50, lr=0.001):
    model = LSTM_Model(input_size=1, hidden_size=64, num_layers=2, output_size=5).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    patience = 10
    patience_counter = 0
    best_state = None
    
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        
        # Validação
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val)
            val_loss = criterion(val_outputs, y_val).item()
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    return model, best_val_loss

print('Classes definidas!')

In [ ]:
# Testar diferentes tamanhos de janela
window_sizes = [10, 20, 30, 50, 100]
results_windows = []

for window in window_sizes:
    print(f'\nTestando janela de {window} dias...')
    
    # Criar sequências
    X, y = create_sequences(df_bovespa['close'].values, n_lags=window, horizon=5)
    
    # Reshape para (samples, timesteps, features)
    X = X.reshape(-1, window, 1)
    
    # Normalização
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    
    X_flat = X.reshape(-1, window)
    X_scaled = scaler_X.fit_transform(X_flat).reshape(-1, window, 1)
    y_scaled = scaler_y.fit_transform(y)
    
    # Particionamento temporal
    train_size = int(0.7 * len(X_scaled))
    val_size = int(0.15 * len(X_scaled))
    
    X_train = torch.FloatTensor(X_scaled[:train_size]).to(device)
    y_train = torch.FloatTensor(y_scaled[:train_size]).to(device)
    X_val = torch.FloatTensor(X_scaled[train_size:train_size+val_size]).to(device)
    y_val = torch.FloatTensor(y_scaled[train_size:train_size+val_size]).to(device)
    X_test = torch.FloatTensor(X_scaled[train_size+val_size:]).to(device)
    y_test = y[train_size+val_size:]
    
    # Treinar
    model, val_loss = train_lstm(X_train, y_train, X_val, y_val, n_epochs=50)
    
    # Prever no teste
    model.eval()
    with torch.no_grad():
        y_pred_scaled = model(X_test).cpu().numpy()
    
    y_pred = scaler_y.inverse_transform(y_pred_scaled)
    
    # Calcular RMSE por horizonte
    rmses = []
    for h in range(5):
        rmse = np.sqrt(mean_squared_error(y_test[:, h], y_pred[:, h]))
        rmses.append(rmse)
    
    mean_rmse = np.mean(rmses)
    results_windows.append({
        'Janela': window,
        'Val Loss': val_loss,
        'RMSE Médio': mean_rmse
    })
    
    print(f'  RMSE Médio: {mean_rmse:.2f}')

# Resultados
results_df = pd.DataFrame(results_windows)
print("\n=== RESULTADOS ===")
print(results_df.to_string(index=False))

# Melhor janela
best_window = results_df.loc[results_df['RMSE Médio'].idxmin(), 'Janela']
print(f"\nMelhor tamanho de janela: {best_window} dias")

In [ ]:
# Visualização
plt.figure(figsize=(10, 6))
plt.plot(results_df['Janela'], results_df['RMSE Médio'], 'bo-', linewidth=2, markersize=8)
plt.xlabel('Tamanho da Janela (dias)')
plt.ylabel('RMSE Médio')
plt.title('Desempenho da LSTM vs Tamanho da Janela de Entrada')
plt.grid(True, alpha=0.3)
plt.xticks(window_sizes)
plt.tight_layout()
plt.show()

print("\nObservações:")
print('- Janelas muito pequenas (10) podem não capturar padrões suficientes')
print('- Janelas muito grandes (100) podem introduzir ruído e overfitting')
print('- O tamanho ótimo depende da frequência dos dados e dos padrões temporais')

### Discussão Exercício 6.1

O tamanho ótimo da janela depende de:
- **Frequência dos dados**: dados diários precisam de janelas maiores que dados mensais
- **Memória da série**: séries com dependências de longo prazo (ex: mortalidade) precisam de janelas maiores
- **Tamanho da amostra**: janelas muito grandes reduzem o número de sequências disponíveis

Regra prática: comece com 2-3 ciclos da periodicidade dominante da série.

---

## Exercício 6.2: Comparação LSTM vs GRU

**Enunciado:** Compare diretamente LSTM e GRU em uma mesma tarefa de previsão de mortalidade. Mantenha todos os hiperparâmetros idênticos (hidden size, número de camadas, learning rate) e diferencie apenas a arquitetura de célula. Meça: (a) tempo de treinamento, (b) número de parâmetros, (c) MAE de teste.

In [ ]:
# =============================================================================
# EXERCÍCIO 6.2: Comparação LSTM vs GRU
# =============================================================================

# Simulação de dados de mortalidade (HMD)
np.random.seed(42)
anos = np.arange(1950, 2021)
n_anos = len(anos)

# Taxa de mortalidade com tendência de queda
tendencia = 0.02 - 0.0002 * (anos - 1950)
mortalidade = tendencia + np.random.normal(0, 0.002, n_anos)
mortalidade = np.maximum(mortalidade, 0.005)

df_mort = pd.DataFrame({
    'mortalidade': mortalidade
}, index=anos)

print(f'Dados de mortalidade: {len(df_mort)} anos')
print(df_mort.head())

In [ ]:
# Classes LSTM e GRU
class LSTMComparison(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=0.2
        )
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()

class GRUComparison(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, output_size=1):
        super().__init__()
        self.gru = nn.GRU(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=0.2
        )
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :]).squeeze()

# Preparar dados
def prepare_data(n_lags=10):
    X, y = [], []
    data = df_mort['mortalidade'].values
    for i in range(n_lags, len(data)):
        X.append(data[i-n_lags:i])
        y.append(data[i])
    
    X = np.array(X).reshape(-1, n_lags, 1)
    y = np.array(y)
    
    # Normalização
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    
    X_flat = X.reshape(-1, n_lags)
    X_scaled = scaler_X.fit_transform(X_flat).reshape(-1, n_lags, 1)
    y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()
    
    # Split
    train_size = int(0.7 * len(X_scaled))
    
    X_train = torch.FloatTensor(X_scaled[:train_size]).to(device)
    y_train = torch.FloatTensor(y_scaled[:train_size]).to(device)
    X_test = torch.FloatTensor(X_scaled[train_size:]).to(device)
    y_test = y[train_size:]
    
    return X_train, y_train, X_test, y_test, scaler_y

X_train, y_train, X_test, y_test, scaler_y = prepare_data()
print('Dados preparados!')

In [ ]:
# Função para treinar e avaliar
def train_and_benchmark(model, X_train, y_train, X_test, y_test, scaler_y, n_epochs=100):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # Medir tempo de treinamento
    start_time = time.time()
    
    best_loss = float('inf')
    patience = 15
    patience_counter = 0
    best_state = None
    
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
    
    training_time = time.time() - start_time
    epochs_trained = epoch + 1
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    # Prever no teste
    model.eval()
    with torch.no_grad():
        y_pred_scaled = model(X_test).cpu().numpy()
    
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    mae = mean_absolute_error(y_test, y_pred)
    
    return {
        'training_time': training_time,
        'epochs': epochs_trained,
        'mae': mae
    }

# Criar modelos
lstm_model = LSTMComparison(input_size=1, hidden_size=64, num_layers=2).to(device)
gru_model = GRUComparison(input_size=1, hidden_size=64, num_layers=2).to(device)

# Contar parâmetros
lstm_params = sum(p.numel() for p in lstm_model.parameters())
gru_params = sum(p.numel() for p in gru_model.parameters())

print(f'LSTM: {lstm_params:,} parâmetros')
print(f'GRU:  {gru_params:,} parâmetros')
print(f'GRU tem {100*(1-gru_params/lstm_params):.1f}% menos parâmetros que LSTM\n')

In [ ]:
# Treinar ambos
print('Treinando LSTM...')
lstm_results = train_and_benchmark(lstm_model, X_train, y_train, X_test, y_test, scaler_y)

print('Treinando GRU...')
gru_results = train_and_benchmark(gru_model, X_train, y_train, X_test, y_test, scaler_y)

# Comparar
print("\n=== COMPARAÇÃO LSTM vs GRU ===")
print(f"\n{'Métrica':<20} {'LSTM':<15} {'GRU':<15} {'Diferença':<15}")
print("-" * 65)
print(f"{'Parâmetros':<20} {lstm_params:<15,} {gru_params:<15,} {f'{100*(gru_params-lstm_params)/lstm_params:.1f}%':<15}")
print(f"{'Tempo de Treino (s)':<20} {lstm_results['training_time']:<15.3f} {gru_results['training_time']:<15.3f} {f'{100*(gru_results['training_time']-lstm_results['training_time'])/lstm_results['training_time']:.1f}%':<15}")
print(f"{'Épocas':<20} {lstm_results['epochs']:<15} {gru_results['epochs']:<15} {gru_results['epochs']-lstm_results['epochs']:<15}")
print(f"{'MAE Teste':<20} {lstm_results['mae']:<15.6f} {gru_results['mae']:<15.6f} {f'{100*(gru_results['mae']-lstm_results['mae'])/lstm_results['mae']:.1f}%':<15}")

print("\nConclusões:")
print('- GRU tem menos parâmetros (mais eficiente)')
print('- GRU é geralmente mais rápido de treinar')
print('- Desempenho preditivo é tipicamente similar')
print('- GRU é preferível quando há limitação computacional')

---

## Exercício 6.3: Arquiteturas Empilhadas (Stacked)

**Enunciado:** Implemente uma arquitetura empilhada (*stacked*) LSTM com 1, 2 e 3 camadas. Avalie o impacto da profundidade no desempenho de previsão de séries temporais econômicas.

In [ ]:
# =============================================================================
# EXERCÍCIO 6.3: Arquiteturas Empilhadas
# =============================================================================

# Simulação de série econômica (PIB)
np.random.seed(42)
trimestres = pd.date_range('2000-01-01', '2023-12-31', freq='Q')
n_tri = len(trimestres)

tendencia = np.linspace(100, 200, n_tri)
ciclo = 10 * np.sin(2 * np.pi * np.arange(n_tri) / 40)
ruido = np.random.normal(0, 3, n_tri)
pib = tendencia + ciclo + ruido

df_pib = pd.DataFrame({'pib': pib}, index=trimestres)

print(f'Dados PIB: {len(df_pib)} trimestres')
print(df_pib.head())

In [ ]:
# LSTM com diferentes números de camadas
class StackedLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size=1):
        super().__init__()
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=0.2 if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()

# Preparar dados
X_pib, y_pib = create_sequences(df_pib['pib'].values, n_lags=12, horizon=1)
X_pib = X_pib.reshape(-1, 12, 1)
y_pib = y_pib.flatten()

scaler_X_pib = MinMaxScaler()
scaler_y_pib = MinMaxScaler()

X_pib_flat = X_pib.reshape(-1, 12)
X_pib_scaled = scaler_X_pib.fit_transform(X_pib_flat).reshape(-1, 12, 1)
y_pib_scaled = scaler_y_pib.fit_transform(y_pib.reshape(-1, 1)).flatten()

train_size = int(0.7 * len(X_pib_scaled))
val_size = int(0.15 * len(X_pib_scaled))

X_train_pib = torch.FloatTensor(X_pib_scaled[:train_size]).to(device)
y_train_pib = torch.FloatTensor(y_pib_scaled[:train_size]).to(device)
X_val_pib = torch.FloatTensor(X_pib_scaled[train_size:train_size+val_size]).to(device)
y_val_pib = torch.FloatTensor(y_pib_scaled[train_size:train_size+val_size]).to(device)
X_test_pib = torch.FloatTensor(X_pib_scaled[train_size+val_size:]).to(device)
y_test_pib = y_pib[train_size+val_size:]

print('Dados PIB preparados!')

In [ ]:
# Testar diferentes profundidades
num_layers_list = [1, 2, 3]
results_depth = []

for num_layers in num_layers_list:
    print(f'\nTreinando LSTM com {num_layers} camada(s)...')
    
    model = StackedLSTM(input_size=1, hidden_size=64, num_layers=num_layers).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    
    # Treinar
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience = 15
    patience_counter = 0
    
    for epoch in range(100):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_pib)
        loss = criterion(outputs, y_train_pib)
        loss.backward()
        optimizer.step()
        
        # Validação
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_pib)
            val_loss = criterion(val_outputs, y_val_pib).item()
        
        train_losses.append(loss.item())
        val_losses.append(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
    
    model.load_state_dict(best_state)
    
    # Avaliar no teste
    model.eval()
    with torch.no_grad():
        y_pred_scaled = model(X_test_pib).cpu().numpy()
    
    y_pred = scaler_y_pib.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    mae = mean_absolute_error(y_test_pib, y_pred)
    
    results_depth.append({
        'Camadas': num_layers,
        'Parâmetros': n_params,
        'Épocas': len(train_losses),
        'Val Loss': best_val_loss,
        'MAE Teste': mae,
        'train_losses': train_losses,
        'val_losses': val_losses
    })
    
    print(f'  Parâmetros: {n_params:,}, MAE: {mae:.4f}')

# Resultados
results_depth_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['train_losses', 'val_losses']} for r in results_depth])
print("\n=== RESULTADOS ===")
print(results_depth_df.to_string(index=False))

In [ ]:
# Visualizar curvas de aprendizado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, result in enumerate(results_depth):
    axes[0].plot(result['train_losses'], label=f"{result['Camadas']} camada(s)"),
    axes[1].plot(result['val_losses'], label=f"{result['Camadas']} camada(s)"),

axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE Loss (Treino)')
axes[0].set_title('Curvas de Treinamento')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Época')
axes[1].set_ylabel('MSE Loss (Validação)')
axes[1].set_title('Curvas de Validação')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Detectar overfitting
print("\n=== ANÁLISE DE OVERFITTING ===")
for result in results_depth:
    final_train = result['train_losses'][-1]
    final_val = result['val_losses'][-1]
    gap = final_val - final_train
    print(f"{result['Camadas']} camada(s): gap treino-val = {gap:.6f} ({'Overfitting' if gap > 0.01 else 'OK'})")

### Discussão Exercício 6.3

Conclusões sobre profundidade:
- **1 camada**: Pode subajustar (underfitting) em séries complexas
- **2 camadas**: Geralmente o ponto ótimo para a maioria das séries
- **3+ camadas**: Risco de overfitting, especialmente com dados limitados

Recomendação: Comece com 2 camadas e aumente apenas se houver evidência de underfitting.

---

## Conclusão

Este notebook apresentou soluções para os exercícios do Capítulo 6:

1. **Tamanho da janela**: Deve ser calibrado empiricamente; janelas muito grandes podem degradar desempenho
2. **LSTM vs GRU**: GRU é mais eficiente computacionalmente com desempenho similar
3. **Profundidade**: 2 camadas são geralmente suficientes; mais camadas aumentam risco de overfitting

Próximos passos recomendados:
- Experimentar arquiteturas bidirecionais
- Implementar regularização (dropout, weight decay)
- Testar em dados reais do mercado financeiro brasileiro